# Pneumonia Detection — Transfer Learning (MobileNetV2)
Two-phase strategy: **Phase 1** frozen base → **Phase 2** fine-tune top layers.

## 1. Imports & GPU Check

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import Model, layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.metrics import confusion_matrix, classification_report

sys.path.insert(0, os.path.abspath('..'))

gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'None — running on CPU')

## 2. Configuration

In [ ]:
DATA_DIR   = '../data/raw/chest_xray'
TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
VAL_DIR    = os.path.join(DATA_DIR, 'val')
TEST_DIR   = os.path.join(DATA_DIR, 'test')
MODEL_PATH = '../models/pneumonia_mobilenetv2.keras'
PLOTS_DIR  = '../results/plots'

os.makedirs('../models', exist_ok=True)
os.makedirs(PLOTS_DIR,   exist_ok=True)

IMAGE_SIZE         = (224, 224)
BATCH_SIZE         = 32
CLASSES            = ['NORMAL', 'PNEUMONIA']
TL_EPOCHS_FROZEN   = 10     # Phase 1: head warm-up only
TL_EPOCHS_FINETUNE = 30     # Phase 2: fine-tune top layers
TL_LR_FROZEN       = 1e-3   # Higher LR — base is frozen
TL_LR_FINETUNE     = 1e-5   # Very low LR — preserve pretrained weights
TL_UNFREEZE_FROM   = 100    # Unfreeze MobileNetV2 layers from this index onward

print('Dataset folders:', os.listdir(DATA_DIR))

## 3. Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
eval_datagen = ImageDataGenerator(rescale=1./255)

common = dict(target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
              color_mode='grayscale', class_mode='binary')

train_gen = train_datagen.flow_from_directory(TRAIN_DIR, shuffle=True,  **common)
val_gen   = eval_datagen.flow_from_directory(VAL_DIR,   shuffle=False, **common)
test_gen  = eval_datagen.flow_from_directory(TEST_DIR,  shuffle=False, **common)

print(f'Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')
print(f'Class indices: {train_gen.class_indices}')

## 4. Model — MobileNetV2 + Custom Head
Grayscale images are repeated across 3 channels so MobileNetV2 (pretrained on RGB ImageNet) can accept them.

In [ ]:
def build_transfer_model(trainable_base=False):
    inputs = layers.Input(shape=(*IMAGE_SIZE, 1))

    # Grayscale → RGB: repeat channel 3 times
    x = layers.Lambda(lambda t: tf.repeat(t, 3, axis=-1), name='gray_to_rgb')(inputs)

    # MobileNetV2 pretrained on ImageNet
    base = tf.keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3), include_top=False, weights='imagenet'
    )

    if trainable_base:
        # Phase 2: freeze early layers, unfreeze top layers
        for layer in base.layers[:TL_UNFREEZE_FROM]:
            layer.trainable = False
        for layer in base.layers[TL_UNFREEZE_FROM:]:
            layer.trainable = True
    else:
        base.trainable = False  # Phase 1: entire base frozen

    x = base(x, training=trainable_base)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='pneumonia_mobilenetv2')


def make_callbacks(path):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            path, monitor='val_accuracy', save_best_only=True, verbose=1
        ),
    ]

## 5. Phase 1 — Train Head Only (Frozen Base)

In [ ]:
model = build_transfer_model(trainable_base=False)
model.compile(
    optimizer=tf.keras.optimizers.Adam(TL_LR_FROZEN),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

train_gen.reset(); val_gen.reset()
history_p1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=TL_EPOCHS_FROZEN,
    callbacks=make_callbacks(MODEL_PATH)
)

## 6. Phase 2 — Fine-Tune Top Layers (Low LR)

In [ ]:
# Rebuild with top layers unfrozen, carry over Phase 1 weights
model_ft = build_transfer_model(trainable_base=True)
model_ft.set_weights(model.get_weights())

model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(TL_LR_FINETUNE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

train_gen.reset(); val_gen.reset()
history_p2 = model_ft.fit(
    train_gen,
    validation_data=val_gen,
    epochs=TL_EPOCHS_FINETUNE,
    callbacks=make_callbacks(MODEL_PATH)
)

## 7. Combined Training Curves (Phase 1 + Phase 2)

In [ ]:
combined   = {k: history_p1.history[k] + history_p2.history[k] for k in history_p1.history}
phase1_end = len(history_p1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (tk, vk), title in zip(
    axes,
    [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
    ['Accuracy', 'Loss']
):
    ax.plot(combined[tk], label=f'Train {title}')
    ax.plot(combined[vk], label=f'Val {title}')
    ax.axvline(phase1_end - 1, color='gray', linestyle='--', label='Fine-tune start')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'tl_training_curves.png'), dpi=150)
plt.show()

## 8. Evaluate on Test Set

In [ ]:
test_gen.reset()
tl_loss, tl_acc = model_ft.evaluate(test_gen, verbose=1)
print(f'\nTest Loss     : {tl_loss:.4f}')
print(f'Test Accuracy : {tl_acc:.4f}')

## 9. Confusion Matrix, Precision, Recall & F1-Score

In [ ]:
test_gen.reset()
y_true = test_gen.classes
y_prob = model_ft.predict(test_gen, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print('\n── Classification Report ──────────────────────────────────────')
print(classification_report(y_true, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.title('Confusion Matrix — MobileNetV2')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'tl_confusion_matrix.png'), dpi=150)
plt.show()

## 10. Predict on a Single Image

In [ ]:
def predict_single(img_path, threshold=0.5):
    img = load_img(img_path, color_mode='grayscale', target_size=IMAGE_SIZE)
    arr = np.expand_dims(img_to_array(img) / 255.0, axis=0)
    prob  = model_ft.predict(arr, verbose=0)[0][0]
    label = CLASSES[int(prob >= threshold)]
    plt.imshow(img, cmap='gray')
    plt.title(f'Prediction: {label}  ({prob:.2%})', fontsize=12)
    plt.axis('off'); plt.show()
    return label, float(prob)


SAMPLE_IMAGE = os.path.join(TEST_DIR, 'PNEUMONIA',
                             os.listdir(os.path.join(TEST_DIR, 'PNEUMONIA'))[0])
label, conf = predict_single(SAMPLE_IMAGE)
print(f'Result → {label}  |  Confidence: {conf:.2%}')

## 11. Save / Load Model

In [ ]:
# Best checkpoint already saved by ModelCheckpoint.
# To reload:
# model_ft = tf.keras.models.load_model(MODEL_PATH)
print(f'Best model saved at: {MODEL_PATH}')